In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [3]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [4]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データの分析を行うにあたり、次のような観点から傾向を考察します。

### 1. カテゴリー別売上傾向
- **食品**: 売上は比較的安定している可能性がありますが、各商品の単価や数量を元に、利益率を計算することで、どの食品が利益をもたらしているかを特定できます。
- **衣服**: 高価格帯の商品の存在から、売上は変動が大きくなる傾向があります。一部の商品は高単価で販売されているため、大きな利益を生むことが期待できます。
- **日用品**: もし日用品がこのデータセットに含まれているとすれば、比較的安定した需要があり、リピート購入が期待できる商品が多いかもしれません。

### 2. 商品別分析
- 各商品の売上数量と単価を元に、販売最適化の観点から、どの商品のプロモーションやマーケティング施策が効果的か調べることができます。
- 利益率（単価 - 原価）を計算し、どの商品の販売が最も効率的かを分析することが重要です。

### 3. 時間的傾向
- 売上日別にデータを集計し、時間の経過に伴う売上の増減を把握します。特定の月や季節に売上が伸びる商品やカテゴリがあるかどうか分析します。
- 例えば、季節的なトレンド（衣服が冬に売れる等）や特定のイベント（バレンタイン、クリスマス）に伴う売上変動も検討する必要があります。

### 4. 利益分析
- 売上高から原価を引き、総利益を算出することで、収益性を評価できます。また、これを商品別、カテゴリー別に行なうことでどの領域に注力すべきか判断できます。

### 5. クロス集計
- 商品間の関連性を調べ、特定の商品が他の商品と一緒に購入される傾向があるかどうかを分析します。これにより、クロスプロモーションの機会を探ることができます。

### 6. 競合分析
- 同様のデータを業界全体で比較し、競合他社に対する相対的な位置付けを行うことも重要です。この情報はマーケティング戦略の見直しに役立ちます。

### 結論
これらの詳細な分析に基づいて、今後のマーケティング戦略、プロモーション計画、及び在庫管理の改善が期待できます。データを活用した意思決定を通じて、企業の成長を促進するための戦略的なインサイトを得ることができます。具体的なデータの可視化（グラフ、ダッシュボード）も助けになるでしょう。


In [5]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0                  売上データの分析を行うにあたり、次のような観点から傾向を考察します。
1                                                    
2                                   ### 1. カテゴリー別売上傾向
3   - **食品**: 売上は比較的安定している可能性がありますが、各商品の単価や数量を元に、利...
4   - **衣服**: 高価格帯の商品の存在から、売上は変動が大きくなる傾向があります。一部の商...
5   - **日用品**: もし日用品がこのデータセットに含まれているとすれば、比較的安定した需要...
6                                                    
7                                        ### 2. 商品別分析
8   - 各商品の売上数量と単価を元に、販売最適化の観点から、どの商品のプロモーションやマーケティ...
9      - 利益率（単価 - 原価）を計算し、どの商品の販売が最も効率的かを分析することが重要です。
10                                                   
11                                       ### 3. 時間的傾向
12  - 売上日別にデータを集計し、時間の経過に伴う売上の増減を把握します。特定の月や季節に売上が...
13  - 例えば、季節的なトレンド（衣服が冬に売れる等）や特定のイベント（バレンタイン、クリスマス...
14                                                   
15                                        ### 4. 利益分析
16  - 売上高から原価を引き、総利益を算出することで、収益性を評価できます。また、これを商品別、...
17                          

In [6]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

In [7]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。
